# Task 6 — Phase 2: Full Evaluation
**MirrorSpeech** | Person 2 (Phase 2)

**Run locally on Mac (Apple M5, CPU)**

This notebook implements:
1. **Load checkpoint** — Whisper + LoRA + ContentHead from `task5_lcma_augmentation.pt`
2. **L2-ARCTIC evaluation** — WER + CER per accent (5 classes: Indian, Mandarin, Korean, Arabic, Native)
3. **EdAcc evaluation** — WER + CER on unseen accents (skipped if not downloaded)
4. **Save results** — `task6_eval_results.json` to checkpoints folder

**Local paths (Scripts/ folder):**
- Splits: `Scripts /splits/`
- Checkpoint: `Scripts /checkpoints/task5_lcma_augmentation.pt`
- Output: `Scripts /checkpoints/task6_eval_results.json`

---
## Cell 1 — Mount Google Drive

In [1]:
import subprocess, sys
pkgs = ['torch', 'torchaudio', 'transformers', 'peft', 'jiwer', 'soundfile']
for pkg in pkgs:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
print('All packages available.')

All packages available.


---
## Cell 2 — Install Packages

In [2]:
# No pip install needed — packages checked in Cell 1
print('Packages ready.')

Packages ready.


---
## Cell 3 — Imports + Device

In [3]:
import os, json
import pathlib
from collections import defaultdict

import torch
import torch.nn as nn
import torchaudio
from torch.utils.data import Dataset, DataLoader

from transformers import WhisperProcessor, WhisperForConditionalGeneration
from peft import get_peft_model, LoraConfig, TaskType
from jiwer import wer, cer

import warnings
warnings.filterwarnings('ignore')

# Force CPU — MPS runs OOM with Whisper-small
device = torch.device('cpu')
DEVICE = device

print(f'PyTorch    : {torch.__version__}')
print(f'Torchaudio : {torchaudio.__version__}')
print(f'Device     : {DEVICE}')

PyTorch    : 2.2.2
Torchaudio : 2.2.2
Device     : cpu


---
## Cell 4 — Set Drive Paths

In [4]:
BASE_DIR     = pathlib.Path.cwd()
if not (BASE_DIR / 'splits').exists():
    BASE_DIR = BASE_DIR / 'Scripts '

SPLITS_DIR   = str(BASE_DIR / 'splits')
EXTRACT_PATH = str(BASE_DIR / 'l2arctic')
LIBRI_DIR    = str(BASE_DIR / 'librispeech')
CKPT_DIR     = str(BASE_DIR / 'checkpoints')
RESULTS_PATH = str(BASE_DIR / 'checkpoints' / 'task6_eval_results.json')
EDACC_DIR    = str(BASE_DIR / 'edacc')

os.makedirs(CKPT_DIR, exist_ok=True)

print(f'Splits      : {SPLITS_DIR}')
print(f'L2-ARCTIC   : {EXTRACT_PATH}')
print(f'LibriSpeech : {LIBRI_DIR}')
print(f'Checkpoints : {CKPT_DIR}')
print(f'Results out : {RESULTS_PATH}')

for f in ['train.json', 'val.json', 'test.json', 'config.json']:
    status = '✅' if os.path.exists(f'{SPLITS_DIR}/{f}') else '❌ MISSING'
    print(f'  {f}: {status}')

Splits      : /Users/shreya/SJSU-Github/Deep_Learning_Project/Scripts /splits
L2-ARCTIC   : /Users/shreya/SJSU-Github/Deep_Learning_Project/Scripts /l2arctic
LibriSpeech : /Users/shreya/SJSU-Github/Deep_Learning_Project/Scripts /librispeech
Checkpoints : /Users/shreya/SJSU-Github/Deep_Learning_Project/Scripts /checkpoints
Results out : /Users/shreya/SJSU-Github/Deep_Learning_Project/Scripts /checkpoints/task6_eval_results.json
  train.json: ✅
  val.json: ✅
  test.json: ✅
  config.json: ✅


---
## Cell 5 — Extract L2-ARCTIC (skip if already extracted)

In [5]:
# L2-ARCTIC is already extracted locally — just verify
import glob
wav_count = len(glob.glob(f'{EXTRACT_PATH}/*/wav/*.wav'))
if wav_count > 0:
    print(f'✅ L2-ARCTIC ready at {EXTRACT_PATH} ({wav_count:,} wav files)')
else:
    print(f'❌ L2-ARCTIC not found at {EXTRACT_PATH}')
    print('Make sure Scripts /l2arctic/ folder has speaker subfolders.')

✅ L2-ARCTIC ready at /Users/shreya/SJSU-Github/Deep_Learning_Project/Scripts /l2arctic (20,044 wav files)


---
## Cell 6 — Load Splits + Fix Paths

In [6]:
with open(f'{SPLITS_DIR}/config.json') as f:
    config = json.load(f)

ACCENT_MAP         = {int(k): v for k, v in config['id_to_accent'].items()}
NUM_ACCENT_CLASSES = int(config['num_accent_classes'])
ACCENT_ID_TO_NAME  = {int(k): v.capitalize() for k, v in config['id_to_accent'].items()}

print('Accent map:', ACCENT_ID_TO_NAME)

def load_and_fix_paths(split_name):
    records = json.load(open(f'{SPLITS_DIR}/{split_name}.json'))
    for r in records:
        orig = r['wav_path'].replace('\\', '/')
        if 'librispeech' in orig.lower():
            tail = orig.split('librispeech/', 1)[-1]
            r['wav_path'] = os.path.join(LIBRI_DIR, tail)
        else:
            r['wav_path'] = os.path.join(
                EXTRACT_PATH, r['speaker'], 'wav', os.path.basename(orig)
            )
    return records

test_records = load_and_fix_paths('test')
print(f'\nTest set : {len(test_records):,} samples')

from collections import Counter
for aid, cnt in sorted(Counter(r['accent_id'] for r in test_records).items()):
    print(f'  {ACCENT_ID_TO_NAME[aid]:10s} (id={aid}): {cnt}')

missing = sum(1 for r in test_records if not os.path.exists(r['wav_path']))
print(f'\nPath check: {len(test_records)-missing}/{len(test_records)} files found')

Accent map: {0: 'Indian', 1: 'Mandarin', 2: 'Korean', 3: 'Arabic', 4: 'Native'}

Test set : 1,855 samples
  Indian     (id=0): 457
  Mandarin   (id=1): 451
  Korean     (id=2): 335
  Arabic     (id=3): 340
  Native     (id=4): 272

Path check: 1855/1855 files found


---
## Cell 7 — Define Model Architecture (must match Task 2)

In [7]:
# ── Content Projection Head (Task 2) ────────────────────────────────────
class ContentHead(nn.Module):
    """Linear(768 → 256) + LayerNorm + ReLU  →  [B, T, 256]"""
    def __init__(self, encoder_dim=768, content_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(encoder_dim, content_dim),
            nn.LayerNorm(content_dim),
            nn.ReLU()
        )
    def forward(self, x):
        return self.net(x)

print('ContentHead defined  →  Linear(768→256) + LayerNorm + ReLU')

ContentHead defined  →  Linear(768→256) + LayerNorm + ReLU


---
## Cell 8 — Load Whisper + LoRA + ContentHead from Checkpoint

In [8]:
# Try Task 5 checkpoint first (Phase 2), fall back to Task 2 (Phase 1)
CKPT_PATH_T5 = f'{CKPT_DIR}/task5_lcma_augmentation.pt'
CKPT_PATH_T2 = f'{CKPT_DIR}/task2_phase1.pt'

if os.path.exists(CKPT_PATH_T5):
    CKPT_PATH = CKPT_PATH_T5
    print(f'Loading Task 5 checkpoint (Phase 2): {CKPT_PATH}')
else:
    CKPT_PATH = CKPT_PATH_T2
    print(f'Task 5 not found, falling back to Task 2: {CKPT_PATH}')

print(f'File exists: {os.path.exists(CKPT_PATH)}')

ckpt = torch.load(CKPT_PATH, map_location=DEVICE)
print(f'Checkpoint keys: {list(ckpt.keys())}')

# ── Whisper base ──────────────────────────────────────────────────────
whisper = WhisperForConditionalGeneration.from_pretrained('openai/whisper-small')

# ── Inject LoRA (same config as Task 2/5) ────────────────────────────
lora_r     = ckpt.get('lora_r', 16)
lora_alpha = ckpt.get('lora_alpha', 32)

lora_cfg = LoraConfig(
    r              = lora_r,
    lora_alpha     = lora_alpha,
    lora_dropout   = 0.1,
    target_modules = ['q_proj', 'v_proj'],
    task_type      = TaskType.FEATURE_EXTRACTION,
)
whisper = get_peft_model(whisper, lora_cfg)
whisper.load_state_dict(ckpt['whisper_lora_state'], strict=False)
whisper = whisper.to(DEVICE)
whisper.eval()

# ── ContentHead ───────────────────────────────────────────────────────
content_head = ContentHead(
    encoder_dim = ckpt.get('encoder_dim', 768),
    content_dim = ckpt.get('content_dim', 256)
).to(DEVICE)
content_head.load_state_dict(ckpt['content_head_state'])
content_head.eval()

# ── Processor ─────────────────────────────────────────────────────────
processor = WhisperProcessor.from_pretrained('openai/whisper-small')

# ── Print config info ─────────────────────────────────────────────────
cfg = ckpt.get('config', {})
print('\n✅ Checkpoint loaded successfully')
print(f'   Source      : {"Task 5 (Phase 2)" if CKPT_PATH == CKPT_PATH_T5 else "Task 2 (Phase 1)"}')
print(f'   LoRA r={lora_r}, alpha={lora_alpha}')
if cfg:
    print(f'   LCMA start  : {cfg.get("lcma_start", "n/a")}')
    print(f'   LCMA final  : {cfg.get("lcma_final", "n/a")}')
    print(f'   Augmentations: {cfg.get("augmentations", "n/a")}')
    print(f'   β (beta)    : {cfg.get("beta", "n/a")}')

Loading Task 5 checkpoint (Phase 2): /Users/shreya/SJSU-Github/Deep_Learning_Project/Scripts /checkpoints/task5_lcma_augmentation.pt
File exists: True
Checkpoint keys: ['whisper_lora_state', 'content_head_state', 'accent_head_state', 'accent_classifier_state', 'reencoder_state', 'config']


generation_config.json: 0.00B [00:00, ?B/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.



✅ Checkpoint loaded successfully
   Source      : Task 5 (Phase 2)
   LoRA r=16, alpha=32
   LCMA start  : 0.14636053144931793
   LCMA final  : 0.08871012181043625
   Augmentations: ['pitch_shift', 'speed_perturb', 'spec_augment']
   β (beta)    : 0.5


---
## Cell 9 — Dataset + DataLoader

In [9]:
TARGET_SR = 16000

class AccentSpeechDataset(Dataset):
    def __init__(self, records, processor, target_sr=TARGET_SR):
        self.records   = records
        self.processor = processor
        self.target_sr = target_sr

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        r = self.records[idx]
        waveform, sr = torchaudio.load(r['wav_path'])
        if sr != self.target_sr:
            waveform = torchaudio.functional.resample(waveform, sr, self.target_sr)
        if waveform.shape[0] > 1:
            waveform = waveform.mean(0, keepdim=True)
        feats = self.processor(
            waveform.squeeze(0).numpy(),
            sampling_rate=self.target_sr,
            return_tensors='pt'
        ).input_features.squeeze(0)
        return {
            'input_features' : feats,
            'accent_id'      : torch.tensor(r['accent_id'], dtype=torch.long),
            'transcript'     : r['transcript'].lower().strip()
        }

def collate_fn(batch):
    feats       = torch.stack([b['input_features'] for b in batch])
    accent_ids  = torch.stack([b['accent_id']      for b in batch])
    transcripts = [b['transcript'] for b in batch]
    return feats, accent_ids, transcripts

test_dataset = AccentSpeechDataset(test_records, processor)
test_loader  = DataLoader(
    test_dataset, batch_size=4, shuffle=False,
    num_workers=0, collate_fn=collate_fn   # num_workers=0 for local Mac
)

print(f'Test dataset : {len(test_dataset):,} samples')
print(f'Batches      : {len(test_loader)}')

Test dataset : 1,855 samples
Batches      : 464


---
## Cell 10 — Evaluation Function (WER + CER per Accent)

In [10]:
def evaluate(model, dataloader, processor, accent_id_to_name, device, desc='Evaluating'):
    model.eval()
    refs_by_accent = defaultdict(list)
    hyps_by_accent = defaultdict(list)

    with torch.no_grad():
        for batch_idx, (feats, accent_ids, transcripts) in enumerate(dataloader):
            feats    = feats.to(device)
            pred_ids = model.generate(feats, language='en', task='transcribe')
            predictions = processor.batch_decode(pred_ids, skip_special_tokens=True)

            for ref, hyp, aid in zip(transcripts, predictions, accent_ids.tolist()):
                name = accent_id_to_name[aid]
                refs_by_accent[name].append(ref.lower().strip())
                hyps_by_accent[name].append(hyp.lower().strip())

            if (batch_idx + 1) % 50 == 0:
                print(f'  {desc}: {batch_idx+1}/{len(dataloader)} batches done')

    results  = {}
    all_refs = []
    all_hyps = []

    print(f'\n=== {desc} Results ===')
    for accent_name in accent_id_to_name.values():
        refs = refs_by_accent.get(accent_name, [])
        hyps = hyps_by_accent.get(accent_name, [])
        if not refs:
            continue
        w = round(wer(refs, hyps), 4)
        c = round(cer(refs, hyps), 4)
        results[accent_name] = {'wer': w, 'cer': c, 'num_samples': len(refs)}
        all_refs.extend(refs)
        all_hyps.extend(hyps)
        print(f'  {accent_name:10s}  WER={w:.1%}  CER={c:.1%}  ({len(refs)} samples)')

    overall_wer = round(wer(all_refs, all_hyps), 4)
    overall_cer = round(cer(all_refs, all_hyps), 4)
    results['overall'] = {'wer': overall_wer, 'cer': overall_cer, 'num_samples': len(all_refs)}
    print(f'  {"OVERALL":10s}  WER={overall_wer:.1%}  CER={overall_cer:.1%}  ({len(all_refs)} samples)')
    return results

print('evaluate() ready')

evaluate() ready


---
## Cell 11 — Run Evaluation on L2-ARCTIC Test Set
> This is the main evaluation: **5 accent classes** (Indian, Mandarin, Korean, Arabic, Native)

In [11]:
print('Running evaluation on L2-ARCTIC + LibriSpeech test set...')
print(f'Total samples: {len(test_dataset):,}\n')

l2_results = evaluate(
    model            = whisper,
    dataloader       = test_loader,
    processor        = processor,
    accent_id_to_name= ACCENT_ID_TO_NAME,
    device           = DEVICE,
    desc             = 'L2-ARCTIC'
)

print('\n✅ L2-ARCTIC evaluation complete')

Running evaluation on L2-ARCTIC + LibriSpeech test set...
Total samples: 1,855

  L2-ARCTIC: 50/464 batches done
  L2-ARCTIC: 100/464 batches done
  L2-ARCTIC: 150/464 batches done
  L2-ARCTIC: 200/464 batches done
  L2-ARCTIC: 250/464 batches done
  L2-ARCTIC: 300/464 batches done
  L2-ARCTIC: 350/464 batches done
  L2-ARCTIC: 400/464 batches done
  L2-ARCTIC: 450/464 batches done

=== L2-ARCTIC Results ===
  Indian      WER=21.1%  CER=6.5%  (457 samples)
  Mandarin    WER=29.0%  CER=11.6%  (451 samples)
  Korean      WER=24.1%  CER=8.0%  (335 samples)
  Arabic      WER=25.7%  CER=8.8%  (340 samples)
  Native      WER=17.2%  CER=4.3%  (272 samples)
  OVERALL     WER=22.7%  CER=7.4%  (1855 samples)

✅ L2-ARCTIC evaluation complete


---
## Cell 12 — Run Evaluation on EdAcc Dataset (Unseen Accents)
> EdAcc tests generalization to accents **not seen during training**.
> If EdAcc is not yet downloaded, this cell prints a setup guide and skips gracefully.

In [15]:
if not os.path.exists(EDACC_DIR):
    print(f'EdAcc not found at: {EDACC_DIR}')
    print('Skipping EdAcc evaluation.')
    edacc_results = None
else:
    edacc_wavs = sorted(glob.glob(f'{EDACC_DIR}/**/*.wav', recursive=True))
    print(f'EdAcc found: {len(edacc_wavs):,} wav files')
    edacc_records = []
    for wav_path in edacc_wavs:
        txt_path = wav_path.replace('.wav', '.txt')
        transcript = open(txt_path).read().strip() if os.path.exists(txt_path) else ''
        edacc_records.append({'wav_path': wav_path, 'transcript': transcript, 'accent_id': 0})

    edacc_dataset = AccentSpeechDataset(edacc_records, processor)
    edacc_loader  = DataLoader(edacc_dataset, batch_size=4, shuffle=False,
                               num_workers=0, collate_fn=collate_fn)
    refs, hyps = [], []
    whisper.eval()
    with torch.no_grad():
        for feats, _, transcripts in edacc_loader:
            pred_ids = whisper.generate(feats, language='en', task='transcribe')
            predictions = processor.batch_decode(pred_ids, skip_special_tokens=True)
            refs.extend([t.lower().strip() for t in transcripts])
            hyps.extend([p.lower().strip() for p in predictions])

    edacc_wer = round(wer(refs, hyps), 4)
    edacc_cer = round(cer(refs, hyps), 4)
    edacc_results = {'overall': {'wer': edacc_wer, 'cer': edacc_cer, 'num_samples': len(refs)}}
    print(f'EdAcc OVERALL  WER={edacc_wer:.1%}  CER={edacc_cer:.1%}')
    print('✅ EdAcc evaluation complete')

EdAcc not found at: /Users/shreya/SJSU-Github/Deep_Learning_Project/Scripts /edacc
Skipping EdAcc evaluation.


---
## Cell 13 — Print Results Summary

In [16]:
print('=' * 55)
print('TASK 6 — FULL EVALUATION SUMMARY')
print('=' * 55)

print('\n── L2-ARCTIC + LibriSpeech (5 accent classes) ──')
print(f'  {"Accent":10s}  {"WER":>8s}  {"CER":>8s}  {"Samples":>8s}')
print(f'  {"-"*10}  {"-"*8}  {"-"*8}  {"-"*8}')
for accent, m in l2_results.items():
    print(f'  {accent:10s}  {m["wer"]:>7.1%}  {m["cer"]:>7.1%}  {m["num_samples"]:>8,}')

if edacc_results:
    print('\n── EdAcc (Unseen Accents) ──')
    m = edacc_results['overall']
    print(f'  {"OVERALL":10s}  {m["wer"]:>7.1%}  {m["cer"]:>7.1%}  {m["num_samples"]:>8,}')
else:
    print('\n── EdAcc : not yet downloaded (see Cell 12) ──')

print('=' * 55)

TASK 6 — FULL EVALUATION SUMMARY

── L2-ARCTIC + LibriSpeech (5 accent classes) ──
  Accent           WER       CER   Samples
  ----------  --------  --------  --------
  Indian        21.1%     6.5%       457
  Mandarin      29.0%    11.6%       451
  Korean        24.1%     8.0%       335
  Arabic        25.7%     8.8%       340
  Native        17.2%     4.3%       272
  overall       22.7%     7.4%     1,855

── EdAcc : not yet downloaded (see Cell 12) ──


---
## Cell 14 — Save Results to Drive (for Task 8)

In [17]:
output = {
    'model'             : 'openai/whisper-small + LoRA',
    'checkpoint'        : CKPT_PATH,
    'checkpoint_source' : 'Task 5 (Phase 2)' if CKPT_PATH == CKPT_PATH_T5 else 'Task 2 (Phase 1)',
    'num_accent_classes': NUM_ACCENT_CLASSES,
    'l2arctic_results'  : l2_results,
    'edacc_results'     : edacc_results if edacc_results else 'not evaluated',
}

with open(RESULTS_PATH, 'w') as f:
    json.dump(output, f, indent=2)

print(f'✅ Results saved → {RESULTS_PATH}')
print(f'   Checkpoint source : {output["checkpoint_source"]}')
print(f'   Accents evaluated : {list(l2_results.keys())}')

✅ Results saved → /Users/shreya/SJSU-Github/Deep_Learning_Project/Scripts /checkpoints/task6_eval_results.json
   Checkpoint source : Task 5 (Phase 2)
   Accents evaluated : ['Indian', 'Mandarin', 'Korean', 'Arabic', 'Native', 'overall']


---
## Task 6 (Person 2) — Phase 2 Complete

### What you built:
```
1. Loaded task2_phase1.pt  →  Whisper + LoRA + ContentHead
2. Evaluated on L2-ARCTIC test set  →  WER + CER for 5 accents
3. Evaluated on EdAcc  →  WER + CER for unseen accents
4. Saved task6_eval_results.json  →  ready for Task 8
```

### How it connects:
```
Task 2 checkpoint (whisper + LoRA)
        ↓
PERSON 2 (Task 6): evaluate() → WER + CER per accent
        ↓
task6_eval_results.json → Task 8 (Results + Report)
```

### Google Drive output:
```
Results : /content/drive/MyDrive/MirrorSpeech/checkpoints/task6_eval_results.json
```